In [ ]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [ ]:
data = pd.read_csv("../data/adult.csv", index_col=0, na_values="?")
data = data.dropna()

expected_salary = np.where(data["salary K$"].gt(50), ">50K", "<=50K")
salary_is_correct = data["salary"].eq(expected_salary).all()

print(f"Rows after cleaning: {data.shape[0]}")
print(f"Salary columns are consistent: {salary_is_correct}")

# Task 1
Print the count of men and women in the dataset.

In [1]:
gender_counts = data["sex"].value_counts()
print(gender_counts)

# Task 2
Find the average age of men in dataset

In [ ]:
average_age_of_men = data.loc[data["sex"].eq("Male"), "age"].mean()
print(f"Average age of men: {average_age_of_men:.2f}")

# Task 3
Get the percentage of people from Poland (native-country)

In [ ]:
poland_percentage = data["native-country"].eq("Poland").mean() * 100
print(f"People from Poland: {poland_percentage:.2f}%")

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [2]:
age_statistics_by_salary = (
    data.groupby("salary")["age"]
    .describe()
    .loc[:, "mean":"std"]
)
print(age_statistics_by_salary)

# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [3]:
has_higher_education = data["education"].str.fullmatch(
    r"Bachelors|Prof-school|Assoc-acdm|Assoc-voc|Masters|Doctorate"
)
rich_without_higher_education = (
    data["salary"].eq(">50K") & ~has_higher_education
)

print(
    "People without higher education earning >50K: "
    f"{rich_without_higher_education.sum()} "
    f"(exist: {rich_without_higher_education.any()})"
)

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [ ]:
age_statistics_by_education = data.groupby("education")["age"].describe()
print(age_statistics_by_education)

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [5]:
men = data.loc[data["sex"].eq("Male")].copy()
men = men.assign(
    marital_group=np.where(
        men["marital-status"].str.startswith("Married"),
        "Married",
        "Not married",
    )
)
salary_percentages = (
    pd.crosstab(men["marital_group"], men["salary"], normalize="index")
    .mul(100)
    .round(2)
)
print(salary_percentages)
print("Married men are more likely to earn >50K.")

# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [6]:
max_hours_per_week = data["hours-per-week"].max()
people_working_max_hours = data["hours-per-week"].eq(max_hours_per_week).sum()

print(f"Maximum hours per week: {max_hours_per_week}")
print(f"People working that many hours: {people_working_max_hours}")

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [7]:
correlation = data.select_dtypes(include="number").corr()
upper_triangle = correlation.where(
    np.triu(np.ones(correlation.shape), k=1).astype(bool)
)
strongest_pair = upper_triangle.abs().stack().idxmax()
strongest_value = correlation.loc[strongest_pair]

print(
    f"Strongest numeric connection: {strongest_pair[0]} and "
    f"{strongest_pair[1]} (correlation: {strongest_value:.3f})."
)
print(correlation.round(3))